# Groundwater Quality Risk Assessment System

## Data Preprocessing

### Steps
1. Load raw dataset
2. Fix data types
3. Standardize label inconsistencies
4. Drop irrelevant columns
5. Handle missing values
6. Cap outliers
7. Encode categorical features
8. Scale numerical features
9. Save preprocessed dataset
10. Save preprocessing artifacts

## 1. Imports & Load Data

In [15]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, RobustScaler

pd.set_option("display.max_columns", None)

In [16]:
df = pd.read_csv("../data/raw/ground_water_quality_2018_2020_unified.csv")

print(f"Shape: {df.shape}")
df.head()

Shape: (1106, 27)


,year,sno,district,mandal,village,lat_gis,long_gis,gwl,season,pH,EC,TDS,CO3,HCO3,Cl,F,NO3,SO4,Na,K,Ca,Mg,TH,SAR,RSC_meq_L,Classification,Classification_drinking
0,2018,1,ADILABAD,Adilabad,Adilabad,19.668300,78.524700,5.09,postmonsoon 2018,8.28,745,476.80,0.0,220.0,60,0.44,42.276818,46.0,49.0,4.0,48.0,38.896,279.934211,1.273328,-1.198684,C2S1,P.S.
1,2018,2,ADILABAD,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,postmonsoon 2018,8.29,921,589.44,0.0,230.0,80,0.56,100.659091,68.0,42.0,5.0,56.0,63.206,399.893092,0.913166,-3.397862,C3S1,P.S.
2,2018,3,ADILABAD,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,postmonsoon 2018,7.69,510,326.40,0.0,200.0,30,0.66,41.471545,44.0,45.0,2.0,24.0,38.896,219.934211,1.319284,-0.398684,C2S1,P.S.
3,2018,4,ADILABAD,Jainath,Jainath,19.730555,78.640000,5.75,postmonsoon 2018,8.09,422,270.08,0.0,160.0,10,0.58,10.669864,35.0,27.0,1.0,32.0,19.448,159.967105,0.928155,0.000658,C2S1,P.S.
4,2018,5,ADILABAD,Narnoor,Narnoor,19.495665,78.852654,2.15,postmonsoon 2018,8.21,2321,1485.44,0.0,300.0,340,2.56,128.843636,280.0,298.0,5.0,56.0,92.378,519.843750,5.682664,-4.396875,C4S2,P.S.


## 2. Fix Data Types

**Issue:** `pH` is stored as string — must be converted to float.

In [17]:
df["pH"] = pd.to_numeric(df["pH"], errors="coerce")

print("pH dtype after fix:", df["pH"].dtype)
print("pH nulls introduced:", df["pH"].isnull().sum())

pH dtype after fix: float64
pH nulls introduced: 1


## 3. Standardize Label Inconsistencies

**Issue:** `"OG"` and `"O.G"` in `Classification` refer to the same class.

In [18]:
print("Before fix:")
print(df["Classification"].value_counts())

df["Classification"] = df["Classification"].str.strip().replace("O.G", "OG")

print("\nAfter fix:")
print(df["Classification"].value_counts())

Before fix:
Classification
C3S1    696
C2S1    248
C4S1     87
C4S2     36
C3S2     12
C4S4      7
C3S3      6
C4S3      5
C1S1      3
OG        2
O.G       2
C3S4      1
C2S2      1
Name: count, dtype: int64

After fix:
Classification
C3S1    696
C2S1    248
C4S1     87
C4S2     36
C3S2     12
C4S4      7
C3S3      6
C4S3      5
OG        4
C1S1      3
C3S4      1
C2S2      1
Name: count, dtype: int64


## 4. Drop Irrelevant or Redundant Columns

| Column | Reason |
|--------|--------|
| `sno` | Serial number, no predictive value |
| `year` | Redundant — captured in `season` |
| `lat_gis`, `long_gis` | Geographic identifiers only |
| `district`, `mandal`, `village` | High-cardinality strings |
| `TDS` | Near-perfect correlation with EC |
| `TH` | Derived from Ca + Mg |
| `SAR` | Derived from Na, Ca, Mg |
| `RSC_meq_L` | Derived from HCO3 + CO3 |

In [19]:
cols_to_drop = [
    "sno", "year",
    "lat_gis", "long_gis",
    "district", "mandal", "village",
    "TDS",
    "TH",
    "SAR",
    "RSC_meq_L"
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"Shape after dropping: {df.shape}")
print("Remaining columns:", df.columns.tolist())

Shape after dropping: (1106, 16)
Remaining columns: ['gwl', 'season', 'pH', 'EC', 'CO3', 'HCO3', 'Cl', 'F', 'NO3', 'SO4', 'Na', 'K', 'Ca', 'Mg', 'Classification', 'Classification_drinking']


## 5. Handle Missing Values

| Column | Missing | Strategy |
|--------|---------|----------|
| `CO3` | 160 (14.47%) | KNN Imputation |
| `gwl` | 11 (0.99%) | Median Imputation |

In [20]:
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before imputation:
gwl     11
pH       1
CO3    160
dtype: int64


In [21]:
# Median imputation for gwl
gwl_imputer = SimpleImputer(strategy="median")
df["gwl"] = gwl_imputer.fit_transform(df[["gwl"]])

# KNN imputation for CO3 (correlated with other ions)
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

knn_imputer = KNNImputer(n_neighbors=5)
df[numerical_cols] = knn_imputer.fit_transform(df[numerical_cols])

print("Total missing values remaining:", df.isnull().sum().sum())

Total missing values remaining: 0


## 6. Outlier Capping (IQR Method)

Chemical readings can have legitimate extreme values, so we **cap** rather than remove outliers using the 1.5×IQR rule.

In [22]:
chemical_cols = ["EC", "CO3", "HCO3", "Cl", "F", "NO3", "SO4", "Na", "K", "Ca", "Mg", "gwl"]

def cap_outliers_iqr(df, columns):
    df = df.copy()
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_capped = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower=lower, upper=upper)
        print(f"{col:15s} | lower={lower:.2f}, upper={upper:.2f} | capped={n_capped}")
    return df

df = cap_outliers_iqr(df, chemical_cols)

EC              | lower=-555.25, upper=3010.75 | capped=50
CO3             | lower=0.00, upper=0.00 | capped=222
HCO3            | lower=-70.00, upper=650.00 | capped=18
Cl              | lower=-240.00, upper=560.00 | capped=52
F               | lower=-0.58, upper=2.62 | capped=52
NO3             | lower=-99.80, upper=213.60 | capped=76
SO4             | lower=-26.88, upper=82.12 | capped=134
Na              | lower=-88.16, upper=296.09 | capped=68
K               | lower=-4.00, upper=12.00 | capped=131
Ca              | lower=-56.00, upper=200.00 | capped=49
Mg              | lower=-41.33, upper=133.70 | capped=42
gwl             | lower=-10.77, upper=26.13 | capped=32


## 7. Encode Categorical Features

### 7a. Season — One-Hot Encoding

In [23]:
print("Unique seasons:", df["season"].unique())

df = pd.get_dummies(df, columns=["season"], drop_first=True)

season_cols = [c for c in df.columns if c.startswith("season_")]
print("Encoded season columns:", season_cols)

Unique seasons: <StringArray>
['postmonsoon 2018 ', 'post monsoon 2019', 'Post-monsoon 2020']
Length: 3, dtype: str
Encoded season columns: ['season_post monsoon 2019', 'season_postmonsoon 2018 ']


### 7b. Classification (Irrigation) — LabelEncoder

Using `LabelEncoder` to treat irrigation classes as **nominal** labels, avoiding assumptions about distances between classes.

In [24]:
le_irrigation = LabelEncoder()
df["Classification_encoded"] = le_irrigation.fit_transform(df["Classification"])

print("Irrigation class encoding:")
for cls, code in zip(le_irrigation.classes_, range(len(le_irrigation.classes_))):
    print(f"  {cls} -> {code}")

Irrigation class encoding:
  C1S1 -> 0
  C2S1 -> 1
  C2S2 -> 2
  C3S1 -> 3
  C3S2 -> 4
  C3S3 -> 5
  C3S4 -> 6
  C4S1 -> 7
  C4S2 -> 8
  C4S3 -> 9
  C4S4 -> 10
  OG -> 11


### 7c. Classification_drinking — LabelEncoder

In [25]:
print("Unique drinking classes:", df["Classification_drinking"].unique())

le_drinking = LabelEncoder()
df["Classification_drinking_encoded"] = le_drinking.fit_transform(
    df["Classification_drinking"].astype(str)
)

print("\nDrinking class encoding:")
for cls, code in zip(le_drinking.classes_, range(len(le_drinking.classes_))):
    print(f"  {cls} -> {code}")

Unique drinking classes: <StringArray>
['P.S.', 'U.S.', 'MR']
Length: 3, dtype: str

Drinking class encoding:
  MR -> 0
  P.S. -> 1
  U.S. -> 2


In [26]:
# Drop original string label columns
df.drop(columns=["Classification", "Classification_drinking"], inplace=True)

print("Columns after encoding:")
print(df.columns.tolist())

Columns after encoding:
['gwl', 'pH', 'EC', 'CO3', 'HCO3', 'Cl', 'F', 'NO3', 'SO4', 'Na', 'K', 'Ca', 'Mg', 'season_post monsoon 2019', 'season_postmonsoon 2018 ', 'Classification_encoded', 'Classification_drinking_encoded']


## 8. Scale Numerical Features

Using **RobustScaler** — uses median and IQR, better suited than StandardScaler when outliers are present.

In [27]:
target_cols  = ["Classification_encoded", "Classification_drinking_encoded"]
dummy_cols   = [c for c in df.columns if c.startswith("season_")]
exclude      = target_cols + dummy_cols

feature_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]

print("Features to scale:", feature_cols)

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("\nScaling done. Sample stats:")
df[feature_cols].describe().T[["mean", "std", "min", "max"]]

Features to scale: ['gwl', 'pH', 'EC', 'CO3', 'HCO3', 'Cl', 'F', 'NO3', 'SO4', 'Na', 'K', 'Ca', 'Mg']

Scaling done. Sample stats:


,mean,std,min,max
gwl,0.262683,0.739745,-0.624390,2.189973
pH,-0.022439,0.672118,-2.580882,3.786765
EC,0.153922,0.782041,-1.093102,2.054122
CO3,0.000000,0.000000,0.000000,0.000000
HCO3,0.046396,0.700444,-1.444444,2.000000
Cl,0.260127,0.749354,-0.600000,2.150000
F,0.180095,0.789295,-1.112500,2.112500
NO3,0.308344,0.813438,-0.531800,2.193214
SO4,0.331053,0.884682,-0.807339,2.169725
Na,0.205935,0.811517,-0.925687,2.103774


## 9. Save Preprocessed Dataset

In [28]:
print("=== Final Preprocessed Dataset ===")
print(f"Shape  : {df.shape}")
print(f"Nulls  : {df.isnull().sum().sum()}")
df.head()

=== Final Preprocessed Dataset ===
Shape  : (1106, 17)
Nulls  : 0


,gwl,pH,EC,CO3,HCO3,Cl,F,NO3,SO4,Na,K,Ca,Mg,season_post monsoon 2019,season_postmonsoon 2018,Classification_encoded,Classification_drinking_encoded
0,-0.091057,0.610294,-0.487381,0.0,-0.388889,-0.35,-0.6125,0.006552,0.844037,-0.468445,0.2225,-0.250,-0.111111,False,True,1,1
1,-0.089973,0.625000,-0.289961,0.0,-0.333333,-0.25,-0.4625,0.751709,1.651376,-0.541314,0.4725,-0.125,0.444444,False,True,3,1
2,-0.102981,-0.257353,-0.750981,0.0,-0.500000,-0.50,-0.3375,-0.003726,0.770642,-0.510085,-0.2775,-0.625,-0.111111,False,True,1,1
3,-0.019512,0.330882,-0.849692,0.0,-0.722222,-0.60,-0.4375,-0.396860,0.440367,-0.697463,-0.5275,-0.500,-0.555556,False,True,1,1
4,-0.409756,0.507353,1.280426,0.0,0.055556,1.05,2.0375,1.111439,2.169725,2.103774,0.4725,-0.125,1.111111,False,True,8,1


In [29]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/groundwater_ml_ready.csv", index=False)

print("Saved: ../data/processed/groundwater_ml_ready.csv")

Saved: ../data/processed/groundwater_ml_ready.csv


## 10. Save Preprocessing Artifacts

These `.pkl` files are required by the Flask backend to preprocess user input **exactly the same way** as training data.

In [30]:
artifact_dir = "../models/preprocessing"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(knn_imputer,    f"{artifact_dir}/imputer.pkl")
joblib.dump(scaler,         f"{artifact_dir}/robust_scaler.pkl")
joblib.dump(season_cols,    f"{artifact_dir}/season_columns.pkl")
joblib.dump(le_irrigation,  f"{artifact_dir}/irrigation_encoder.pkl")
joblib.dump(le_drinking,    f"{artifact_dir}/drinking_encoder.pkl")
joblib.dump(feature_cols,   f"{artifact_dir}/feature_columns.pkl")

print("Saved preprocessing artifacts:")
for f in os.listdir(artifact_dir):
    print(f"  {artifact_dir}/{f}")

Saved preprocessing artifacts:
  ../models/preprocessing/drinking_encoder.pkl
  ../models/preprocessing/feature_columns.pkl
  ../models/preprocessing/imputer.pkl
  ../models/preprocessing/irrigation_encoder.pkl
  ../models/preprocessing/robust_scaler.pkl
  ../models/preprocessing/season_columns.pkl
